# Import Dependencies

In [57]:
#region
#  1. Data Management (Always first)
%load_ext autoreload
%autoreload 2

# 2. Objects you are actually manipulating in cells
import numpy as np
import h5py
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 3. The Bridge to your script
from bender_functions import Bender
#endregion

DEBUG: Loading bender_functions.py from: c:\Users\jimen\Desktop\Git_YEJ\Bender_PC\Bender3\bender_functions.py
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Where do you want to save the data files?

In [58]:
# Setup configuration
bender = Bender(config_module_name='jimenez_bender_config_A')

# Set up the data folder and base name for the output files
data_folder = r'c:\Users\jimen\Desktop\BenderData\RandomTest'
base_name = '2026-04-01_Polyurethane40A'

#region  Create/show file path AND check configuration summary
bender.outputfile = bender.increment_file_name(f"{data_folder}\\{base_name}.h5") # Create full file path
bender.outputfig = bender.outputfile.replace('.h5', '.png') # Mirror that name for the figure by simply swapping the extension
file_data = {
    "Type": ["HDF5 Data", "PNG Figure"],
    "Full System Path": [bender.outputfile, bender.outputfig]
}
df_files = pd.DataFrame(file_data) # Turn the Dictionary into a DataFrame Instance
pd.set_option('display.max_colwidth', None) # Tell Pandas not to cut off any text in the columns
display(df_files) # Display the Instance

# Print the summary of the Bender instance to check that everything is set up correctly
summary = bender.summary()
#endregion
print(bender.outputfile)

Bender initialized using: jimenez_bender_config_A.py


,Type,Full System Path
0,HDF5 Data,c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A001.h5
1,PNG Figure,c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A001.png


              BENDER SYSTEM SUMMARY               
Config:      jimenez_bender_config_A.py
Device:      Dev1
Motor Port:  port0
Direction:   POSITIVE = LEFT
--------------------------------------------------
Cal File:    FT56491.cal
Sample Rate: 1000.0 Hz
Ramp:        0.25 s
c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A001.h5


## Biometrics: Input these before mounting

In [59]:
bender.update_metadata(
    fishcode = "40A polyurethane rod", 
    segment = "NA",
    fishmass = "NA" ,   # Body mass in grams
    fishlen_TL = 304.8 , # Total length in mm
    fishlen_SL = 304.8, # Standard length in mm

    # Mount-specific dimensions: input these after mounting but BEFORE running the experiment
    xsec_width = 25.4 ,       # mm Cross sectional width of fish between the clamps at the axis of rotation
    xsec_height = 25.4,       # mm Cross sectional height of fish between the clamps at the axis of rotation

    # Mounted biometrics
    dbend = 180 ,     # mm Distance from snout to the center of pressure ?
    dclamp = 52 ,     # mm Distance between the two clamps
    dvert = 47 ,         # mm Vertical distence from the transducer to the center of pressure
    dhoriz = 00          # mm Horizontal distence from the transducer to the center of pressure
)

  Stored: fishcode = 40A polyurethane rod
  Stored: segment = NA
  Stored: fishmass = NA
  Stored: fishlen_TL = 304.8
  Stored: fishlen_SL = 304.8
  Stored: xsec_width = 25.4
  Stored: xsec_height = 25.4
  Stored: dbend = 180
  Stored: dclamp = 52
  Stored: dvert = 47
  Stored: dhoriz = 0


## Optional: if you want to drive bending motion based on muscle strain. 
If you want to drive curvature based on body thickness and red muscle strain, a simple combination can be performed to create inputs for all_curves.

In [60]:
# Use desired muscle strain to calculate the curvature amplitudes for bending motions.
desired_strain_pct = np.array([5,10])  # Desired strain in %
body_thickness_m = bender.xsec_width/1000   # Full thickness of the specimen in METERS. 
desired_curves = ((2 * (desired_strain_pct / 100)) / body_thickness_m) # Formula: Curvature (1/m) = (2 * Strain) / Thickness (m)

# What kind of experiment would you like to run?

In [61]:
bender.update_metadata(
    test_type = "dynamic", # Select your test type!

    # Bending motions
    all_freqs = [.5,5], # Unlimited options for dynamic tests, but for frequency sweep these are start and end frequencies.
    all_curves = [2,4], # For dyanmic, ulimited. For sweeps, pick only one. Enter 'desired_curves' if you want to use the curvature calculated from the desired muscle strain.
    randomize = False,


    cycles_per_step = 5,    # How many times do you want to bend your specimen at each amplitude/frequency?
    n_end_cycles = 2,        # add cycles after last amplitude step
    stim_cycles_in_step = np.array([2,3]), # This array defines which cycles to activate
    # Bending motions for FREQUENCY SWEEP only (these will get ignore if you run a different test type)
    duration = 60,      # sec. How long to you want the whole test to last?
    amplitude_frequency_exponent = 0,   # should be between -1 and 0. Zero is constant amplitude, -1 is constant velocity, -0.5 is right in the middle.

    # Electrical stiulation
    is_stim = False, # True stimulates the muscle. False is for passive tests
    all_stimduties = [0],       # fractions of a cycle
    all_stimphases = [0],      # fractions of a cycle
    stim_pulse_rate = 75, # Hz shouldn't have to change this from 75
    
    # Change these to whatever the stimulator panel is set to.
    S1volts = 10, # Volts
    S2volts = 10, # Volts
    S1pulsedur = 2,          # ms
    S2pulsedur = 2           # ms
)

  Stored: test_type = dynamic
  Stored: all_freqs = [0.5, 5]
  Stored: all_curves = [2, 4]
  Stored: randomize = False
  Stored: cycles_per_step = 5
  Stored: n_end_cycles = 2
  Stored: stim_cycles_in_step = [2 3]
  Stored: duration = 60
  Stored: amplitude_frequency_exponent = 0
  Stored: is_stim = False
  Stored: all_stimduties = [0]
  Stored: all_stimphases = [0]
  Stored: stim_pulse_rate = 75
  Stored: S1volts = 10
  Stored: S2volts = 10
  Stored: S1pulsedur = 2
  Stored: S2pulsedur = 2


## Organize the experimental ouptuts (motion, stimuli, etc)

In [62]:
#region Run the calculation method to generate the sequence attributes. This applies only to some test_types (dynamic, static)
if bender.test_type in ['dynamic', 'static']:
    bender.organize_cycles(
        all_curves=bender.all_curves,
        all_freqs=bender.all_freqs,
        randomize=bender.randomize,
        cycles_per_step=bender.cycles_per_step,
        n_end_cycles=bender.n_end_cycles,
        dclamp=bender.dclamp,
        xsec_width=bender.xsec_width,
        stim_cycles_in_step=bender.stim_cycles_in_step,
        all_stimduties=bender.all_stimduties,
        all_stimphases=bender.all_stimphases,
        stim_pulse_rate=bender.stim_pulse_rate
    )

    # Generate the numbers for the plot
    angle, anglevel, tnorm, t = bender.make_dynamic_cycles(
        bender.period_by_cycle,
        bender.freq_by_cycle, 
        bender.amp_by_cycle
    )
    
    bender.record_motor_signal(t, angle, anglevel, tnorm)

elif bender.test_type in ['sweep']:
    bender.duration = duration # duration of each sweep in seconds
    bender.all_freqs = all_freqs # frequencies of sweep in Hz (should be a list of 2 values: [start, end])
    bender.all_curves = all_curves # amplitude of sweep in degrees
    bender.xsec_width = xsec_width
    bender.amplitude_frequency_exponent = amplitude_frequency_exponent # exponent for how amplitude changes
   
    # Generate the numbers for the plot
    angle, anglevel, tnorm, t = bender.make_frequency_sweep(
        bender.all_freqs, 
        bender.all_curves, 
        bender.amplitude_frequency_exponent, 
        bender.waitbefore
    )
#endregion


## CHECK: Is the experimental sequence what you wanted?

In [63]:
#region Make a table
print("--- Generated Sequence Attributes ---")

# Use the 'organized_' versions so all arrays have the same length
display(pd.DataFrame({
    "freq (Hz)": bender.organized_freqs,           
    "curve (1/m)": bender.organized_curves,       
    "amp (deg)": bender.all_degs,           
    "strain (%)": bender.all_strains * 100,
    "strain rate (%/s)": bender.all_strainrates * 100,
    "duty (%)": bender.organized_stimduties,      
    "phase (%)": bender.organized_stimphases      
}))


#MAKE PLOTS

# First plot: angle/stim plot with shaded stimulus periods
bender.make_stimuli()
fig = go.Figure()
fig.add_trace(go.Scatter(x=bender.t, y=bender.angle, name="Commanded Angle", line=dict(color='black')))

# Add Shading (ONLY if stimulation is active)
if bender.is_stim:
    # Handle Left Side Shading
    if hasattr(bender, 'Lonoff') and bender.Lonoff is not None:
        for onoff in bender.Lonoff:
            fig.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="blue", opacity=0.5, line_width=0, name="Left Stim")

    # Handle Right Side Shading
    if hasattr(bender, 'Ronoff') and bender.Ronoff is not None:
        for onoff in bender.Ronoff:
            fig.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="red", opacity=0.5, line_width=0, name="Right Stim")

fig.update_layout(title="Bending-Activation Preview", xaxis_title="Time (s)", yaxis_title="Angle (deg)")

display(fig)

# Second Plot: Muscle strain over time
kappa_over_time = np.deg2rad(bender.angle) / (bender.dclamp / 1000)  # Formula: curvature = angle_radians / length
strain_pct_over_time = ((kappa_over_time * (bender.xsec_width/1000)) / 2) * 100 # Formula: strain (%)= (curvature * thickness) / 2
fig_strain = go.Figure()

fig_strain.add_trace(go.Scatter(
    x=bender.t, 
    y=strain_pct_over_time, 
    name="Strain (%)", 
    line=dict(color='black')
))

# Add the same shading stim shading
if bender.is_stim:
    if hasattr(bender, 'Lonoff') and bender.Lonoff is not None:
        for onoff in bender.Lonoff:
            fig_strain.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="blue", opacity=0.3, line_width=0)
    if hasattr(bender, 'Ronoff') and bender.Ronoff is not None:
        for onoff in bender.Ronoff:
            fig_strain.add_vrect(x0=onoff[0], x1=onoff[1], fillcolor="red", opacity=0.3, line_width=0)

fig_strain.update_layout(
    title="Strain Over Time Preview", 
    xaxis_title="Time (s)", 
    yaxis_title="Strain (%)"
)
display(fig_strain)
#endregion

--- Generated Sequence Attributes ---


,freq (Hz),curve (1/m),amp (deg),strain (%),strain rate (%/s),duty (%),phase (%)
0,0.5,2,5.958761,2.54,7.979645,0,0
1,5.0,2,5.958761,2.54,79.796453,0,0
2,0.5,4,11.917522,5.08,15.959291,0,0
3,5.0,4,11.917522,5.08,159.592907,0,0


## Input dimensions to calculate Moment of Inertia for Clamps and Specimen

In [64]:
clamp_offset = 20.0 # mm Distance between the rotating clamps front margin and the axis of rotation
front_h, front_w = 160, 30
back_h, back_w = 20, 10 
spec_length = 100.0

# Set the physics for the bender based on the mounting dimensions and specimen dimensions. This will be used to calculate the moment of inertia and other parameters for the bending
bender.set_physics(clamp_offset, front_h, front_w, back_h, back_w, spec_length)


           PHYSICS CONFIGURATION REPORT           
Mode                      | lateral              
Total Rotating MOI        | 1540379.44   | g*mm²
Lever Arm (r)             | 30.00        | mm
Specimen Mass             | 383.50       | g
--------------------------------------------------
TOTAL SYSTEM MASS         | 807.98       | g



## CALCULATE: important measurements to be included in the data file (H5).

In [65]:
#Theoretically not necessary since most values are saved, but simplifies pipeline.  
#test_section_pos = dbend/(fishlen_TL + dclamp + CLAMP_D)   # Position of the bending segment at the axis of rotation (fraction of total length). Assuming the clamp depth (CLAMP_D below) is 20. 


# Make sure to save stim data!! Should look something like this:
# ADD STIMULI (code will ignore if is_stim is False)
# 1. Generate the stimulus signals
#S1stimcmd, S2stimcmd = bender.make_stimuli()

# 2. Extract the Lonoff and Ronoff lists the function just saved
#Lonoff = bender.Lonoff
#Ronoff = bender.Ronoff

# START BENDING!! 

This is the main code block that runs the experiment. It sets up the DAQ, sends the output, records the input, and writes it to the file.

In [66]:
bender.run_experiment(test_type="dynamic")


Data will be saved to: experiment_data_dynamic_000005.h5
📏 Sonometer (Left) Calibrated: 14.86 mm
📏 Sonometer (Right) Calibrated: 14.86 mm


## Save ALL data
Make sure to check the folder to see the files are actually saving there. Otherwise all the data is lost!

In [67]:
with h5py.File(bender.outputfile, 'w') as f:
    # 1. Primary Buckets
    g_meta = f.create_group('01_Metadata')
    g_ts   = f.create_group('02_TimeSeries')
    
    # 2. Sub-folders for organization
    g_cmd = g_ts.create_group('Commands')   
    g_cal = g_ts.create_group('Calibrated') 
    g_raw = g_ts.create_group('Raw')        

    # 3. THE MASTER LOOP
    for key, value in bender.__dict__.items():
        if key.startswith('_') or value is None: 
            continue
            
        try:
            val = np.array(value)
            is_large = val.size > 100
            
            # --- CATEGORY A: High-Frequency Data ---
            if is_large:
                if any(x in key.lower() for x in ['stim', 'cmd', 'plan', 'duty', 'phase']):
                    target = g_cmd
                elif any(x in key.lower() for x in ['force', 'torque', 'moment', 'angle', 'deg']):
                    target = g_cal
                else:
                    target = g_raw
                
                ds_obj = target.create_dataset(key, data=val)

                # Special Case: aidata breakout
                # Using 'input_channel_names' from your config
                if key == 'aidata' and hasattr(bender, 'input_channel_names'):
                    for i, name in enumerate(bender.input_channel_names):
                        g_raw.create_dataset(f"Ch{i}_{name.replace(' ', '_')}", data=val[:, i])

            # --- CATEGORY B: Metadata ---
            else:
                g_meta.attrs[key] = value
                ds_obj = None # Flag for attribute assignment

            # --- 4. SMART UNIT ATTRIBUTION (The "Foolproof" Step) ---
            found_unit = None
            
            # Priority 1: Exact match (e.g., body_thickness -> meters)
            if hasattr(bender, 'units') and key in bender.units:
                found_unit = bender.units[key]
            
            # Priority 2: Pattern match (e.g., width -> mm)
            elif hasattr(bender, 'unit_rules'):
                for pattern, unit in bender.unit_rules.items():
                    if pattern in key.lower():
                        found_unit = unit
                        break

            # Save the unit to HDF5
            if found_unit:
                if is_large:
                    ds_obj.attrs['units'] = found_unit
                else:
                    g_meta.attrs[f"{key}_units"] = found_unit

        except Exception:
            g_meta.attrs[key] = str(value)

    # --- 5. TRACEABILITY ---
    try:
        with open(f"{bender.config_name}.py", 'r') as cfg:
            g_meta.attrs['Config_File_Content'] = cfg.read()
    except: pass

print(f"✅ ARCHIVE SUCCESS: {bender.outputfile} structured and labeled for R.")

✅ ARCHIVE SUCCESS: c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A001.h5 structured and labeled for R.


## QUALITY CONTROL: Plot control signals AND raw torque over time

In [68]:
raw_v = bender.aidata[6,0] # First sample of the sono channel
cal_mm = bender.apply_calibration_sono(raw_v, bender.sono_cal_left)

print(f"Voltage at start: {raw_v:.3f} V")
print(f"Calculated distance: {cal_mm:.3f} mm")

Voltage at start: -10.550 V
Calculated distance: -108.807 mm


In [69]:
#region Make sure to plot the correct bending axis. It depends on sensor mounting and orientation!
fig = make_subplots(rows = 5, cols = 1, shared_xaxes=True)

# --- Row 1: Angle Plot (Encoder vs Command) ---

# 1. Add Encoder Angle (Measured)
fig.add_trace(
    go.Scatter(x=bender.t, y=bender.angle_measured, mode="lines", name="angle_enc", line=dict(color='blue')),
    row=1, col=1
)
# 2. Add Command Angle (Target)
fig.add_trace(
    go.Scatter(x=bender.t, y=bender.angle, mode="lines", name="angle_cmd", line=dict(dash='dash', color='black')),
    row=1, col=1
)


fig.add_trace(
    go.Scatter(x = bender.t, y = bender.sono_left_mm, mode="lines", name="sono1_mm"),
    row=4, col=1)
fig.add_trace(
    go.Scatter(x = bender.t, y = bender.sono_right_mm, mode="lines", name="sono2_mm", line = dict(dash='dash')),
    row=4, col=1)

fig.add_trace(
    go.Scatter(x = bender.t, y = bender.forcetorque[3,:], mode="lines", name="Tx"),
    row=2, col=1)

fig.add_trace(
    go.Scatter(x = bender.t, y = bender.forcetorque[5,:], mode="lines", name="Tz"),
    row=3, col=1)

# --- Row 5: Individual Strain Plot (%) ---

# 1. Define the baseline window
mask = (bender.t >= -0.5) & (bender.t <= 0)
# 2. Calculate initial average lengths for each side
l0_left = bender.sono_left_mm[mask].mean()
l0_right = bender.sono_right_mm[mask].mean()
# 3. Calculate strain as a percentage: ((L - L0) / L0) * 100
strain_left_pct = ((bender.sono_left_mm - l0_left) / l0_left) * 100
strain_right_pct = ((bender.sono_right_mm - l0_right) / l0_right) * 100

# 4. Add both traces to Row 5
fig.add_trace(
    go.Scatter(x=bender.t, y=strain_left_pct, name="Strain Left (%)", line=dict(color='blue')),
    row=5, col=1
)
fig.add_trace(
    go.Scatter(x=bender.t, y=strain_right_pct, name="Strain Right (%)", line=dict(color='red')),
    row=5, col=1
)

# Update axes titles
fig.update_yaxes(title_text="angle (deg)", row=1)
fig.update_yaxes(title_text="Tx (Nm)", row=2)
fig.update_yaxes(title_text="Tz (Nm)", row=3)
fig.update_yaxes(title_text="Distance (mm)", row=4)
fig.update_yaxes(title_text="Strain (%)", row=5)
fig.update_xaxes(title_text="time (s)", row=5)

# 1. Update layout for a clean 5-row PNG export
fig.update_layout(
    height=1500,     # Increased height so 5 plots aren't squashed in the PNG
    width=1000, 
    title_text=bender.outputfile,
    showlegend=True
)

# 2. Display in your Jupyter Notebook
fig.show()

# 3. Save as a high-resolution PNG
# Scale=2 makes the text and lines crisp for reports/papers
fig.write_image(f"{bender.outputfile}.png", engine="kaleido", scale=2)

print(f"✅ Plot displayed and saved as: {bender.outputfile}.png")
#endregion

✅ Plot displayed and saved as: c:\Users\jimen\Desktop\BenderData\RandomTest\2026-04-01_Polyurethane40A001.h5.png


## QUALITY CONTROL: Do the loops like nice?

In [56]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=bender.t, y=bender.aidata[6,:]))
fig.update_yaxes(title_text="torque (Nm)")
fig.update_xaxes(title_text="angle (deg)")